# Exercise 7.3: Tracker material budget

From *Programming in High Energy Particle Physics*, Chapter 7

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch07/ex07_3.ipynb)

Exercise 7.3 Tracker material budget Use Geant4 to estimate the material budget (in radiation lengths) of a typical tracking detector as a function of pseudorapidity. Identify which detector components contribute the most material in the endcap region.

<details><summary>Hint</summary>

Start from Listing 7.11 . Give the beam pipe, the sensor layers, the support structures and the service volumes (cables, cooling) separate logical volumes, and fill one weighted histogram per component, selected by the logical volume of the step.

</details>

### Setup

The Colab plot is a **geometry-based analytic stand-in**, not Geant4. `geant4_pybind` is an optional heavy install that may download large datasets. A local C++ Geant4 installation is needed to transport geantinos through the geometry below.

In [ ]:
import sys
IN_COLAB="google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy matplotlib
INSTALL_GEANT4_BINDINGS=False
if IN_COLAB and INSTALL_GEANT4_BINDINGS:
    %pip install -q geant4_pybind
import numpy as np
import matplotlib.pyplot as plt
rng=np.random.default_rng(42)

### Complete local Geant4 material scan

A geantino is fired uniformly in η and φ. The event CSV records separate beam pipe, sensor, support, and service sums of step length / material radiation length, following Listing 7.11. The model is intentionally generic; it is not CMS geometry.

In [ ]:
%%writefile ex07_3.cc
#include "G4RunManager.hh"
#include "G4VUserDetectorConstruction.hh"
#include "G4VUserPrimaryGeneratorAction.hh"
#include "G4VUserActionInitialization.hh"
#include "G4UserEventAction.hh"
#include "G4UserSteppingAction.hh"
#include "G4NistManager.hh"
#include "G4Box.hh"
#include "G4Tubs.hh"
#include "G4LogicalVolume.hh"
#include "G4PVPlacement.hh"
#include "G4ParticleGun.hh"
#include "G4ParticleTable.hh"
#include "G4SystemOfUnits.hh"
#include "G4Event.hh"
#include "G4Step.hh"
#include "FTFP_BERT.hh"
#include "Randomize.hh"
#include <fstream>
#include <string>
#include <cstdlib>
#include <cmath>
#include <limits>
class Tracker : public G4VUserDetectorConstruction {
public:
  G4VPhysicalVolume* Construct() override {
    auto* nist=G4NistManager::Instance();
    auto* vacuum=nist->FindOrBuildMaterial("G4_Galactic");
    auto* be=nist->FindOrBuildMaterial("G4_Be");
    auto* si=nist->FindOrBuildMaterial("G4_Si");
    auto* carbon=nist->FindOrBuildMaterial("G4_C");
    auto* cu=nist->FindOrBuildMaterial("G4_Cu");
    auto* world=new G4LogicalVolume(new G4Box("world",1*m,1*m,1.2*m),vacuum,"world");
    auto* pv=new G4PVPlacement(nullptr,{},world,"world",nullptr,false,0);
    auto place=[&](const char* name,G4Material* mat,double r1,double r2,double hz,double z,int id) {
      auto* lv=new G4LogicalVolume(new G4Tubs(name,r1,r2,hz,0,360*deg),mat,name);
      new G4PVPlacement(nullptr,G4ThreeVector(0,0,z),lv,name,world,false,id);
    };
    place("beam",be,2.5*cm,2.58*cm,40*cm,0,0);
    int id=0;
    for(double r:{5.,10.,20.,30.}) {
      place("sensor",si,r*cm,(r+0.03)*cm,40*cm,0,id++);
      place("support",carbon,(r+0.5)*cm,(r+0.7)*cm,40*cm,0,id++);
    }
    for(int sign:{-1,1}) {
      for(double z:{45.,70.}) {
        place("sensor",si,5*cm,45*cm,0.015*cm,sign*z*cm,id++);
        place("support",carbon,5*cm,45*cm,0.10*cm,sign*(z+2)*cm,id++);
      }
      for(double z:{55.,80.})
        place("service",cu,5*cm,45*cm,0.25*cm,sign*z*cm,id++);
    }
    return pv;
  }
};
class GeantinoGun : public G4VUserPrimaryGeneratorAction {
  G4ParticleGun gun{1};
public:
  GeantinoGun() {
    gun.SetParticleDefinition(G4ParticleTable::GetParticleTable()->FindParticle("geantino"));
    gun.SetParticleEnergy(1*GeV);
    gun.SetParticlePosition(G4ThreeVector());
  }
  void GeneratePrimaries(G4Event* e) override {
    const double eta=-2.5+5.0*G4UniformRand();
    const double phi=2*CLHEP::pi*G4UniformRand();
    const double theta=2*std::atan(std::exp(-eta));
    gun.SetParticleMomentumDirection(G4ThreeVector(std::sin(theta)*std::cos(phi),std::sin(theta)*std::sin(phi),std::cos(theta)));
    gun.GeneratePrimaryVertex(e);
  }
};
class BudgetEvent : public G4UserEventAction {
  std::ofstream out;
public:
  double eta=std::numeric_limits<double>::quiet_NaN();
  double sums[4]={0,0,0,0};
  explicit BudgetEvent(const std::string& path):out(path) {out << "eta,beam_x0,sensor_x0,support_x0,service_x0\n";}
  void BeginOfEventAction(const G4Event*) override {eta=std::numeric_limits<double>::quiet_NaN();for(double& x:sums)x=0;}
  void EndOfEventAction(const G4Event*) override {
    out << eta;
    for(double x:sums)out << ',' << x;
    out << '\n';
  }
};
class BudgetStep : public G4UserSteppingAction {
  BudgetEvent* evt;
public:
  explicit BudgetStep(BudgetEvent* e):evt(e){}
  void UserSteppingAction(const G4Step* step) override {
    const auto* pre=step->GetPreStepPoint();
    if(std::isnan(evt->eta)) evt->eta=pre->GetMomentumDirection().eta();
    const auto name=pre->GetTouchableHandle()->GetVolume()->GetLogicalVolume()->GetName();
    int index=-1;
    if(name=="beam")index=0;
    if(name=="sensor")index=1;
    if(name=="support")index=2;
    if(name=="service")index=3;
    if(index>=0)evt->sums[index]+=step->GetStepLength()/pre->GetMaterial()->GetRadlen();
  }
};
class Actions : public G4VUserActionInitialization {
  std::string path;
public:
  explicit Actions(std::string p):path(p){}
  void Build() const override {
    SetUserAction(new GeantinoGun);
    auto* evt=new BudgetEvent(path);SetUserAction(evt);
    SetUserAction(new BudgetStep(evt));
  }
};
int main(int argc,char** argv) {
  if(argc<3)return 2; // events, CSV
  G4Random::setTheSeed(42);
  auto* rm=new G4RunManager;
  rm->SetUserInitialization(new Tracker);
  rm->SetUserInitialization(new FTFP_BERT);
  rm->SetUserInitialization(new Actions(argv[2]));
  rm->Initialize();rm->BeamOn(std::atoi(argv[1]));delete rm;
}


In [ ]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(ex07_3 LANGUAGES CXX)
find_package(Geant4 REQUIRED)
include(${Geant4_USE_FILE})
add_executable(ex07_3 ex07_3.cc)
target_link_libraries(ex07_3 ${Geant4_LIBRARIES})


Local commands after extracting the files:

```bash
cmake -S . -B build
cmake --build build -j2
./build/ex07_3 10000 budget.csv
```

Bin `budget.csv` in η; divide each component’s summed weights by the number of geantinos in that bin. This is equivalent to the weighted-histogram / count-histogram ratio in Listing 7.11.
If CMake cannot locate an optional dependency in an existing Geant4 installation, a direct build is also possible with `c++ ex07_N.cc -o ex07_N $(geant4-config --cflags --libs)` (replace `N` with this exercise number and configure the library path for that installation).

### Colab stand-in exercise

In [ ]:
eta=np.linspace(-2.5,2.5,301)
# TODO: compute barrel path length t/sin(theta) and disk path length t/|cos(theta)|.
# TODO: divide each component path by its radiation length X0.
component_x0={}

In [ ]:
assert eta.shape==(301,)
print("Self-check: supports/services should become prominent toward the barrel–endcap transition.")

Plot each component and total material in X₀ versus η. Compare qualitative trends with the book’s CMS example without treating the toy as CMS geometry.